# Metagenomics Diversity Analysis

This notebook performs diversity analysis for microbiome data:
- Alpha diversity (Shannon, Simpson, Chao1)
- Taxonomic composition (stacked bar plots)
- Top taxa identification

**Input**: Taxonomy abundance table (Kraken/MetaPhlAn/BIOM format)
**Outputs**: Diversity plots and statistics

In [ ]:
# Parameters
TOP_N_TAXA = 15
MIN_ABUNDANCE = 0.01  # Minimum relative abundance to show
TAX_LEVEL = 'genus'  # phylum, class, order, family, genus, species
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load abundance data
# Support multiple formats: Kraken report, MetaPhlAn, generic TSV
def load_abundance_data(filepath):
    """Load and normalize abundance data from various formats."""
    df = pd.read_csv(filepath, sep='\t')
    
    # Detect format
    if 'relative_abundance' in df.columns.str.lower():
        # MetaPhlAn format
        return df
    elif 'percentage' in df.columns.str.lower() or 'fraction' in df.columns.str.lower():
        # Kraken/Bracken format
        return df
    else:
        # Generic format - assume first column is taxon, rest are samples
        return df

input_path = get(0, format='path')
df = load_abundance_data(input_path)
print(f"Loaded data with {len(df)} rows")
df.head()

In [ ]:
# Prepare abundance matrix
# Identify taxon column and abundance columns
taxon_col = None
for col in ['name', 'taxon', 'taxonomy', 'clade_name', 'species', '#OTU ID']:
    if col in df.columns or col.lower() in df.columns.str.lower():
        taxon_col = col if col in df.columns else df.columns[df.columns.str.lower() == col.lower()][0]
        break

if taxon_col is None:
    taxon_col = df.columns[0]
    print(f"Using first column as taxon: {taxon_col}")

# Find abundance columns (numeric)
abundance_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Taxon column: {taxon_col}")
print(f"Abundance columns: {abundance_cols[:5]}..." if len(abundance_cols) > 5 else f"Abundance columns: {abundance_cols}")

## 1. Alpha Diversity

In [ ]:
def calculate_alpha_diversity(abundances):
    """Calculate alpha diversity metrics."""
    # Normalize to proportions
    p = abundances / abundances.sum()
    p = p[p > 0]  # Remove zeros
    
    # Shannon diversity
    shannon = -np.sum(p * np.log(p))
    
    # Simpson diversity (1 - D)
    simpson = 1 - np.sum(p ** 2)
    
    # Observed species (richness)
    richness = len(p)
    
    # Chao1 estimator (simplified)
    singletons = np.sum(abundances == 1)
    doubletons = np.sum(abundances == 2)
    if doubletons > 0:
        chao1 = richness + (singletons ** 2) / (2 * doubletons)
    else:
        chao1 = richness + singletons * (singletons - 1) / 2
    
    return {
        'shannon': shannon,
        'simpson': simpson,
        'richness': richness,
        'chao1': chao1
    }

# Calculate diversity for each sample
diversity_results = []
for col in abundance_cols:
    div = calculate_alpha_diversity(df[col].values)
    div['sample'] = col
    diversity_results.append(div)

div_df = pd.DataFrame(diversity_results)
div_df = div_df.set_index('sample')
print("Alpha Diversity Metrics:")
div_df

In [ ]:
# Plot alpha diversity
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = ['shannon', 'simpson', 'richness', 'chao1']
titles = ['Shannon Diversity', 'Simpson Diversity', 'Observed Species', 'Chao1 Richness']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for ax, metric, title, color in zip(axes.flat, metrics, titles, colors):
    if len(div_df) > 1:
        div_df[metric].plot(kind='bar', ax=ax, color=color, edgecolor='white')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    else:
        ax.bar([0], div_df[metric].values, color=color)
        ax.set_xticks([0])
        ax.set_xticklabels(div_df.index)
    ax.set_title(title)
    ax.set_ylabel('Value')

plt.tight_layout()
put(fig, 'alpha_diversity.png')
plt.show()

## 2. Taxonomic Composition

In [ ]:
# Calculate relative abundance
abundance_matrix = df.set_index(taxon_col)[abundance_cols].copy()

# Normalize to relative abundance (percentage)
rel_abundance = abundance_matrix.div(abundance_matrix.sum(axis=0), axis=1) * 100

# Get top taxa by mean abundance
mean_abundance = rel_abundance.mean(axis=1).sort_values(ascending=False)
top_taxa = mean_abundance.head(TOP_N_TAXA).index.tolist()

# Aggregate rest as "Other"
top_rel = rel_abundance.loc[top_taxa].copy()
other = rel_abundance.loc[~rel_abundance.index.isin(top_taxa)].sum()
top_rel.loc['Other'] = other

print(f"Top {TOP_N_TAXA} taxa selected")
top_rel.head()

In [ ]:
# Stacked bar plot
fig, ax = plt.subplots(figsize=(max(12, len(abundance_cols) * 0.8), 8))

# Create color palette
colors = plt.cm.tab20(np.linspace(0, 1, len(top_rel)))

# Plot stacked bars
top_rel.T.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)

ax.set_ylabel('Relative Abundance (%)')
ax.set_xlabel('Sample')
ax.set_title('Taxonomic Composition')
ax.legend(title='Taxon', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
put(fig, 'taxonomic_composition.png')
plt.show()

## 3. Top Taxa Heatmap

In [ ]:
# Heatmap of top taxa
fig, ax = plt.subplots(figsize=(max(10, len(abundance_cols) * 0.5), max(8, TOP_N_TAXA * 0.4)))

# Use log scale for better visualization
heatmap_data = np.log10(top_rel.iloc[:-1] + 0.01)  # Exclude 'Other', add pseudocount

sns.heatmap(heatmap_data, 
            cmap='YlOrRd', 
            ax=ax,
            cbar_kws={'label': 'log10(Relative Abundance %)'})

ax.set_title(f'Top {TOP_N_TAXA} Taxa Abundance Heatmap')
ax.set_xlabel('Sample')
ax.set_ylabel('Taxon')

plt.tight_layout()
put(fig, 'taxa_heatmap.png')
plt.show()

## 4. Summary

In [ ]:
# Generate summary
summary = f"""
# Metagenomics Diversity Analysis Summary

## Dataset Overview
- Total taxa: {len(df):,}
- Samples analyzed: {len(abundance_cols)}

## Alpha Diversity (mean across samples)
- Shannon Index: {div_df['shannon'].mean():.2f} (± {div_df['shannon'].std():.2f})
- Simpson Index: {div_df['simpson'].mean():.3f} (± {div_df['simpson'].std():.3f})
- Observed Species: {div_df['richness'].mean():.0f} (± {div_df['richness'].std():.0f})
- Chao1 Richness: {div_df['chao1'].mean():.0f} (± {div_df['chao1'].std():.0f})

## Dominant Taxa (Top 5 by mean abundance)
"""

for i, (taxon, abundance) in enumerate(mean_abundance.head(5).items(), 1):
    summary += f"{i}. {taxon}: {abundance:.2f}%\n"

print(summary)
put(summary, 'diversity_summary.md')

In [ ]:
# Save diversity table
put(div_df.reset_index(), 'alpha_diversity_table.tsv')

print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - alpha_diversity.png")
print("  - taxonomic_composition.png")
print("  - taxa_heatmap.png")
print("  - diversity_summary.md")
print("  - alpha_diversity_table.tsv")